# VocalCoach Colab Training

Two-stage training to solve the pitch/technique multi-task gradient conflict.

**Why this matters:** Joint training from epoch 1 causes technique gradients (~10x stronger
than pitch at epoch 1) to immediately capture the backbone, collapsing VDR to <5%.
Two-stage training with backbone freezing prevents this:

- **Stage 1** — pitch + VAD only, 50 epochs. Backbone builds clean pitch representations (target VDR ~70%).
- **Stage 2** — resume from stage 1, add technique head. Backbone is **frozen** for the first
  20 epochs so only `head_technique` trains. After epoch 20 the backbone unfreezes for joint fine-tuning.

**Run cells in order.** Cells 1–4 are setup (re-run at the start of every new session).
Then pick an experiment section.

---
**Before starting:** upload `NanoPitch_data.zip` to `My Drive/musicalAI/vocalCoach/` on Google Drive.
```bash
cd ~/NanoPitch-MusicalAI
zip -j -1 -v NanoPitch_data.zip \
    data/clean.npz data/noise.npz data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz
```

## Cell 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Set batch size and workers based on GPU
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH_SIZE = 64 if vram_gb > 30 else 32 if vram_gb > 15 else 16
NUM_WORKERS = 8
print(f"\nUsing batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS}")

## Cell 2 — Mount Drive and extract data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'
RUNS_DIR   = f'{DRIVE_ROOT}/NanoPitch-runs'
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs('/content/data/vocalset', exist_ok=True)

print("Extracting NanoPitch_data.zip...")
with zipfile.ZipFile(f'{DRIVE_ROOT}/NanoPitch_data.zip', 'r') as z:
    for name in ['clean.npz', 'noise.npz', 'test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/')
    for name in ['technique_train.npz', 'technique_test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/vocalset/')

print("\nExtraction complete.")
!ls -lh /content/data/
!ls -lh /content/data/vocalset/

## Cell 3 — Clone repo and install dependencies

In [ ]:
!git clone https://github.com/YOUR_USERNAME/NanoPitch-MusicalAI /content/NanoPitch-MusicalAI
%cd /content/NanoPitch-MusicalAI
!pip install -r requirements.txt --quiet
print("Setup complete.")

## Cell 4 — Verify data loads correctly

In [ ]:
import numpy as np

clean = np.load('/content/data/clean.npz')
print(f"clean.npz:              {list(clean.keys())}")
print(f"  mel shape:            {clean['mel'].shape}")
print(f"  clips:                {clean['lengths'].shape[0]}")

tech = np.load('/content/data/vocalset/technique_train.npz')
print(f"\ntechnique_train.npz:    {list(tech.keys())}")
print(f"  clips:                {tech['lengths'].shape[0]}")

test = np.load('/content/data/test.npz')
print(f"\ntest.npz:               {list(test.keys())}")
print(f"  clips:                {test['clips'].shape[0]}")

---
## TCN — Stage 1: pitch + VAD only (50 epochs)

No technique data. Builds clean pitch representations before technique gradients are introduced.
Target by epoch 50: VDR > 60%, RPA > 96%.

Checkpoints write directly to Drive — a session disconnect loses at most one epoch.

**A100 optimisations vs local runs:**
- `--batch-size 64` (4× local) — A100 has 80 GB VRAM, model uses <1 GB
- `--num-workers 8` (4× local default) — reduces CPU data-loading bottleneck
- `--seq-len 600` — longer sequences = more GPU work per step

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/tcn_stage1_pitchonly \
    --epochs 50 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

## TCN — Stage 2: frozen backbone (epochs 51–70) then joint fine-tuning (71–100)

Resume from stage 1. Backbone frozen for 20 epochs — only `head_technique` trains.
After epoch 20 the backbone unfreezes for joint fine-tuning of all heads.

Can run in a new session — re-run cells 1–4 first, checkpoint is already on Drive.

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/tcn_stage2_technique \
    --epochs 100 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --freeze-backbone-epochs 20 \
    --patience 0 \
    --resume /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/tcn_stage1_pitchonly/checkpoints/best_loss.pth

---
## Conformer — Stage 1: pitch + VAD only (50 epochs)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage1_pitchonly \
    --epochs 50 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

## Conformer — Stage 2: frozen backbone (epochs 51–70) then joint fine-tuning (71–100)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage2_technique \
    --epochs 100 --batch-size 64 --num-workers 8 --seq-len 600 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --freeze-backbone-epochs 20 \
    --patience 0 \
    --resume /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage1_pitchonly/checkpoints/best_loss.pth

---
## Evaluate a completed run

Copies the run from Drive to local disk first (faster I/O than reading from Drive directly).
Change `RUN_NAME` to the run you want to evaluate.

In [ ]:
RUN_NAME   = "tcn_stage2_technique"  # change to the run you want to evaluate
DRIVE_ROOT = '/content/drive/MyDrive/musicalAI/vocalCoach'

import shutil, os
os.makedirs(f'/content/runs/{RUN_NAME}', exist_ok=True)
shutil.copytree(
    f'{DRIVE_ROOT}/NanoPitch-runs/{RUN_NAME}',
    f'/content/runs/{RUN_NAME}',
    dirs_exist_ok=True
)
print("Copied. Running evaluation...")

!python vocalcoach/evaluate.py \
    --checkpoint /content/runs/$RUN_NAME/checkpoints/best_metric.pth \
    --data-dir /content/data \
    --technique-dir /content/data/vocalset